Tests data ingestion is the same as original code by comparing questionnaire, microdata and paradata output.

In [21]:
from pathlib import Path
import pandas as pd
from typing import Dict, Any, List, Optional, Tuple
import numpy as np
from collections import Counter
import math
from pandas.api import types as ptypes
import pyarrow.parquet as pq

from rissk.config import DATA_DIR, RAW_DATA_DIR, PROCESSED_DATA_DIR, INTERIM_DATA_DIR, PROJ_ROOT

In [2]:
SURVEY = "hies2024"

In [3]:
# original files
df_para = pd.read_parquet(INTERIM_DATA_DIR.joinpath("paradata.parquet"))
df_questionnaire = pd.read_parquet(PROCESSED_DATA_DIR.joinpath("questionnaire.parquet"))
df_microdata = pd.read_parquet(PROCESSED_DATA_DIR.joinpath("microdata.parquet"))

In [4]:
# Kedro pipeline outputs
df_para_kedro = pd.read_parquet(PROJ_ROOT.joinpath("rissk_kedro", "data", SURVEY, "latest", "20_INTERIM", "paradata.parquet"))
df_questionnaire_kedro = pd.read_parquet(PROJ_ROOT.joinpath("rissk_kedro", "data", SURVEY, "latest", "30_PROCESSED", "questionnaire.parquet"))
df_microdata_kedro = pd.read_parquet(PROJ_ROOT.joinpath("rissk_kedro", "data", SURVEY, "latest", "30_PROCESSED", "microdata.parquet"))

In [5]:
# Comparison utility inserted into notebook
def _dtype_map(dseries: pd.Series) -> Dict[str, str]:
    return {col: str(dtype) for col, dtype in dseries.items()}


def _find_candidate_key(df_a: pd.DataFrame, df_b: pd.DataFrame, common_cols: List[str]) -> Optional[str]:
    # Prefer obvious id-like columns
    candidates = [c for c in common_cols if any(k in c.lower() for k in ['id', 'uuid', 'key', 'interview'])]
    # fall back to all common cols
    candidates = candidates + [c for c in common_cols if c not in candidates]
    for c in candidates:
        try:
            if df_a[c].is_unique and df_b[c].is_unique:
                return c
        except Exception:
            continue
    return None


def _is_numeric_series(s: pd.Series) -> bool:
    return ptypes.is_numeric_dtype(s.dtype)


def _try_convert_numeric(val):
    try:
        if isinstance(val, str):
            # Check if it looks like a list
            if val.startswith('[') and val.endswith(']'):
                # It's a list string. Compare as list?
                # For now, just return as is (string) comparison
                return val
        return float(val)
    except (ValueError, TypeError):
        return val


def _compare_elementwise(a: pd.Series, b: pd.Series, atol: float, rtol: float) -> np.ndarray:
    """Return boolean mask where True indicates a != b (treating NaNs as equal).
    Works for numeric (uses isclose) and non-numeric (stringified) series.
    Has special handling for numeric-string mismatch (e.g. '1' vs '1.0').
    """
    # Align lengths assumed equal and indexes aligned
    
    # helper for mixed types
    def smart_compare(val_a, val_b):
        if val_a == val_b:
            return True
        # checks for nan
        try:
            if np.isnan(val_a) and np.isnan(val_b):
                return True
        except:
            pass
            
        # Try numeric conversion
        try:
            fa = float(val_a)
            fb = float(val_b)
            if np.isnan(fa) and np.isnan(fb):
                return True
            return np.isclose(fa, fb, atol=atol, rtol=rtol)
        except (ValueError, TypeError):
            # If conversion fails, strict string comparison was already done at start
            return str(val_a) == str(val_b)

    # 1. If numeric series, use vectorized numeric comparison
    if _is_numeric_series(a) and _is_numeric_series(b):
        # convert to float with NaN preserved
        a_f = a.astype(float)
        b_f = b.astype(float)
        # both NaN -> equal
        both_nan = a_f.isna() & b_f.isna()
        # use isclose (NaNs produce False) then invert and mask NaNs
        close = np.isclose(a_f.fillna(np.nan), b_f.fillna(np.nan), atol=atol, rtol=rtol, equal_nan=True)
        neq = ~close
        neq[both_nan.values] = False
        return neq
    
    # 2. For object/mixed series, use element-wise smart comparison
    # This is slower but necessary for '1' vs '1.0' in object columns
    # We can optimize by first checking string equality
    a_s = a.fillna('__NA__').astype(str)
    b_s = b.fillna('__NA__').astype(str)
    
    # Boolean mask of string mismatches
    neq_mask = (a_s != b_s).to_numpy()
    
    # If no string mismatches, we are done
    if not neq_mask.any():
        return neq_mask
        
    # For the mismatches, try numeric comparison
    # Get indices of mismatches
    mismatch_indices = np.where(neq_mask)[0]
    
    # Use list comprehension for the mismatched subset
    a_vals = a.iloc[mismatch_indices].values
    b_vals = b.iloc[mismatch_indices].values
    
    resolved_mask = []
    for va, vb in zip(a_vals, b_vals):
        # We need to check if they are "close enough"
        is_equal = smart_compare(va, vb)
        # If is_equal is True, then we flag it as FALSE (no difference)
        resolved_mask.append(not is_equal)
        
    # Update the neq_mask
    neq_mask[mismatch_indices] = resolved_mask
    
    return neq_mask


def compare_parquet_files(df_a: pd.DataFrame, df_b: pd.DataFrame, check: Optional[str] = None, atol: float = 1e-9, rtol: float = 1e-8) -> Tuple[bool, Dict[str, Any]]:
    """
    Robust comparison of two DataFrames produced by different pipelines.

    - Basic checks (shape, columns, dtypes) always computed.
    - `check` can be None, 'cells' or 'rows'.
    - For 'cells': attempts label-aligned comparison using a detected unique key column
      (preferred if present in both tables), otherwise aligns by index intersection when possible,
      otherwise falls back to positional/overlap comparison. Reports columns that have any differing
      cells and total number of differing cells compared.

    Returns: (same: bool, details: dict)
    """
    if not isinstance(df_a, pd.DataFrame) or not isinstance(df_b, pd.DataFrame):
        raise TypeError('compare_parquet_files expects pandas DataFrame inputs')

    details: Dict[str, Any] = {}

    # Basic metadata
    shape_equal = df_a.shape == df_b.shape
    details['shape'] = {'equal': bool(shape_equal), 'shape_a': df_a.shape, 'shape_b': df_b.shape}

    cols_a = list(df_a.columns)
    cols_b = list(df_b.columns)
    only_a = [c for c in cols_a if c not in cols_b]
    only_b = [c for c in cols_b if c not in cols_a]
    different_columns = list(dict.fromkeys(only_a + only_b))
    columns_equal = len(different_columns) == 0
    details['columns'] = {'different_columns': different_columns, 'equal': bool(columns_equal), 'only_in_a': only_a, 'only_in_b': only_b}

    dtypes_a = _dtype_map(df_a.dtypes)
    dtypes_b = _dtype_map(df_b.dtypes)
    common = [c for c in cols_a if c in cols_b]
    dtype_mismatch = [c for c in common if dtypes_a.get(c) != dtypes_b.get(c)]
    dtypes_equal = len(dtype_mismatch) == 0
    details['dtypes'] = {'mismatched_columns': dtype_mismatch, 'equal': bool(dtypes_equal)}

    same = bool(shape_equal and columns_equal and dtypes_equal)

    # Normalize alias
    if check == 'cell':
        check = 'cells'

    # Automatic key detection (if any)
    candidate_key = _find_candidate_key(df_a, df_b, common)
    details['auto_key'] = candidate_key
    # Determine columns to compare when using a key-based alignment.
    # Exclude the key column(s) from the per-column comparison because
    # setting the key as index will remove it from the DataFrame columns.
    if candidate_key is None:
        cols_to_compare = common
    else:
        if isinstance(candidate_key, (list, tuple)):
            key_list = list(candidate_key)
        else:
            key_list = [candidate_key]
        cols_to_compare = [c for c in common if c not in key_list]
    # choose active columns for comparison depending on whether we will align by key
    cols = cols_to_compare if candidate_key is not None else common

    # CELL-level comparison (position/label depending on alignment)
    if check == 'cells':
        cell_info: Dict[str, Any] = {'checked': True}
        diff_df = None
        if len(common) == 0:
            cell_info['note'] = 'no common columns to compare'
            cell_info['columns_with_differences'] = []
            cell_info['total_cell_differences'] = 0
        else:
            # Prefer key-based alignment
            if candidate_key is not None:
                note = f"aligned by key='{candidate_key}'"
                # set index by key and intersect
                a_k = df_a.set_index(candidate_key)
                b_k = df_b.set_index(candidate_key)
                common_idx = a_k.index.intersection(b_k.index)
                a_al = a_k.loc[common_idx, cols].fillna('__NA__')
                b_al = b_k.loc[common_idx, cols].fillna('__NA__')
                rows_compared = len(common_idx)
            else:
                # try index-based alignment if helpful
                inter_idx = df_a.index.intersection(df_b.index)
                if df_a.index.is_unique and df_b.index.is_unique and len(inter_idx) > 0:
                    note = 'aligned by index intersection'
                    a_al = df_a.reindex(index=inter_idx, columns=common).fillna('__NA__')
                    b_al = df_b.reindex(index=inter_idx, columns=common).fillna('__NA__')
                    rows_compared = len(inter_idx)
                else:
                    # fall back to positional comparison over overlap
                    rows_to_compare = min(len(df_a), len(df_b))
                    note = 'positional comparison over overlapping rows'
                    a_al = df_a.iloc[:rows_to_compare][common].fillna('__NA__').reset_index(drop=True)
                    b_al = df_b.iloc[:rows_to_compare][common].fillna('__NA__').reset_index(drop=True)
                    rows_compared = rows_to_compare

            # perform elementwise comparison with tolerance on numeric cols
            neq_mask = np.zeros((rows_compared, len(cols)), dtype=bool)
            for j, col in enumerate(cols):
                a_col = a_al[col]
                b_col = b_al[col]
                col_neq = _compare_elementwise(a_col, b_col, atol=atol, rtol=rtol)
                neq_mask[:, j] = col_neq

            neq_df = pd.DataFrame(neq_mask, columns=cols)
            cols_with_diff = neq_df.any(axis=0)
            cols_with_diff_names = cols_with_diff[cols_with_diff].index.tolist()
            total_cell_diffs = int(neq_df.values.sum())

            cell_info['columns_with_differences'] = cols_with_diff_names
            cell_info['total_cell_differences'] = total_cell_diffs
            cell_info['rows_compared'] = int(rows_compared)
            cell_info['note'] = note
            
            # Generate difference dataframe if needed
            if total_cell_diffs > 0:
                diff_list = []
                # Finding indices (row, col) of differences
                rows, cols_idx = np.where(neq_mask)
                for r, c in zip(rows, cols_idx):
                    col_name = cols[c]
                    # Get index label if available
                    idx_label = a_al.index[r]
                    val_a = a_al.iloc[r, c]
                    val_b = b_al.iloc[r, c]
                    diff_list.append({
                        'index': idx_label,
                        'column': col_name,
                        'value_a': val_a,
                        'value_b': val_b
                    })
                diff_df = pd.DataFrame(diff_list)

            if total_cell_diffs > 0:
                same = False
        details['cell_compare'] = cell_info
        if diff_df is not None:
            details['diff_df'] = diff_df

    # ROW-level comparison
    if check == 'rows':
        row_info: Dict[str, Any] = {'checked': True}
        if len(common) == 0:
            row_info['note'] = 'no common columns to compare; cannot perform row membership check'
            
        else:
            if candidate_key is not None:
                # compare by key: count keys only in A/B and mismatched rows for common keys
                a_k = df_a.set_index(candidate_key)[cols].fillna('__NA__')
                b_k = df_b.set_index(candidate_key)[cols].fillna('__NA__')
                keys_a = set(a_k.index)
                keys_b = set(b_k.index)
                keys_only_a = keys_a - keys_b
                keys_only_b = keys_b - keys_a
                common_keys = keys_a & keys_b

                # count per-key mismatches
                mismatched_keys = 0
                for k in common_keys:
                    a_row = a_k.loc[k]
                    b_row = b_k.loc[k]
                    # elementwise comparison across common cols
                    neq_any = False
                    for col in cols:
                        if _compare_elementwise(pd.Series([a_row[col]]), pd.Series([b_row[col]]), atol=atol, rtol=rtol)[0]:
                            neq_any = True
                            break
                    if neq_any:
                        mismatched_keys += 1

                row_info['keys_only_in_a'] = int(len(keys_only_a))
                row_info['keys_only_in_b'] = int(len(keys_only_b))
                row_info['mismatched_common_keys'] = int(mismatched_keys)
                row_info['num_rows_different'] = int(len(keys_only_a) + len(keys_only_b) + mismatched_keys)
                row_info['total_rows_a'] = len(df_a)
                row_info['total_rows_b'] = len(df_b)
                row_info['note'] = f"compared by key='{candidate_key}'"

                if row_info['num_rows_different'] > 0:
                    same = False
            else:
                # multiset row comparison on common cols (stringified)
                a_rows = df_a[common].fillna('__NA__').astype(str)
                b_rows = df_b[common].fillna('__NA__').astype(str)
                a_tuples = [tuple(r) for r in a_rows.values]
                b_tuples = [tuple(r) for r in b_rows.values]
                cnt_a = Counter(a_tuples)
                cnt_b = Counter(b_tuples)
                rows_a_not_b = sum(max(cnt_a[k] - cnt_b.get(k, 0), 0) for k in cnt_a)
                rows_b_not_a = sum(max(cnt_b[k] - cnt_a.get(k, 0), 0) for k in cnt_b)
                num_diff = int(rows_a_not_b + rows_b_not_a)

                row_info['rows_in_a_not_in_b'] = int(rows_a_not_b)
                row_info['rows_in_b_not_in_a'] = int(rows_b_not_a)
                row_info['num_rows_different'] = num_diff
                row_info['total_rows_a'] = len(a_tuples)
                row_info['total_rows_b'] = len(b_tuples)
                row_info['note'] = 'multiset row comparison on common columns'

                if num_diff > 0:
                    same = False
        details['row_compare'] = row_info

    details['same'] = bool(same)
    return bool(same), details


In [6]:
_, details = compare_parquet_files(df_para_kedro, df_para, check='cells')


In [7]:
print(details['shape'])
print(details['columns'])
print(details['dtypes'])

{'equal': False, 'shape_a': (78413, 11), 'shape_b': (78413, 27)}
{'different_columns': ['timestamp_local', 'qnr', 'qnr_version', 'qnr_seq', 'variable_name', 'qtype', 'question_type', 'answers', 'question_scope', 'yes_no_view', 'is_filtered_combobox', 'is_integer', 'cascade_from_question_id', 'answer_sequence', 'n_answers', 'question_sequence'], 'equal': False, 'only_in_a': [], 'only_in_b': ['timestamp_local', 'qnr', 'qnr_version', 'qnr_seq', 'variable_name', 'qtype', 'question_type', 'answers', 'question_scope', 'yes_no_view', 'is_filtered_combobox', 'is_integer', 'cascade_from_question_id', 'answer_sequence', 'n_answers', 'question_sequence']}
{'mismatched_columns': ['timestamp_utc', 'tz_offset'], 'equal': False}


In [8]:
details['diff_df']

,index,column,value_a,value_b
0,0,timestamp_utc,2024-10-29T01:17:15.712,2024-10-29 01:17:15.712000
1,0,tz_offset,11:00:00,0 days 11:00:00
2,1,timestamp_utc,2024-10-29T01:17:15.712,2024-10-29 01:17:15.712000
3,1,tz_offset,11:00:00,0 days 11:00:00
4,2,timestamp_utc,2024-10-29T01:17:15.712,2024-10-29 01:17:15.712000
...,...,...,...,...
156821,78410,tz_offset,11:00:00,0 days 11:00:00
156822,78411,timestamp_utc,2024-12-02T01:12:20.744,2024-12-02 01:12:20.744000
156823,78411,tz_offset,11:00:00,0 days 11:00:00
156824,78412,timestamp_utc,2024-12-02T01:12:20.744,2024-12-02 01:12:20.744000


In [9]:
_, details = compare_parquet_files(df_microdata_kedro, df_microdata, check='cells')

In [10]:
print(details['shape'])
print(details['columns'])
print(details['dtypes'])
print(details['cell_compare'])
try:
    display(details['df_diff'])
except:
    print('Cells are the same')

{'equal': False, 'shape_a': (0, 0), 'shape_b': (23127, 41)}
{'different_columns': ['interview__id', 'roster_level', 'variable', 'value', 'filename', 'qnr', 'qnr_version', 'qnr_seq', 'variable_name', 'qtype', 'question_type', 'answers', 'children', 'condition_expression', 'hide_if_disabled', 'featured', 'instructions', 'properties', 'public_key', 'question_scope', 'question_text', 'stata_export_caption', 'variable_label', 'is_timestamp', 'validation_conditions', 'yes_no_view', 'is_filtered_combobox', 'is_integer', 'categories_id', 'title', 'is_roster', 'linked_to_roster_id', 'linked_to_question_id', 'cascade_from_question_id', 'parents', 'answer_sequence', 'n_answers', 'is_linked', 'parent_1', 'parent_2', 'question_sequence'], 'equal': False, 'only_in_a': [], 'only_in_b': ['interview__id', 'roster_level', 'variable', 'value', 'filename', 'qnr', 'qnr_version', 'qnr_seq', 'variable_name', 'qtype', 'question_type', 'answers', 'children', 'condition_expression', 'hide_if_disabled', 'feature

In [11]:
print(df_questionnaire.shape, df_questionnaire_kedro.shape)

(52, 36) (52, 36)


In [12]:
df_questionnaire.columns == df_questionnaire_kedro.columns

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True])

In [13]:
pd.DataFrame({'df_questionnaire': df_questionnaire.dtypes, 'df_questionnaire_kedro': df_questionnaire_kedro.dtypes})

,df_questionnaire,df_questionnaire_kedro
qnr_seq,int64,int64
variable_name,object,object
qtype,object,object
question_type,float64,float64
answers,object,object
children,object,object
condition_expression,object,object
hide_if_disabled,object,object
featured,object,object
instructions,object,object


In [14]:
compare_parquet_files(df_questionnaire_kedro, df_questionnaire, check='cells')

(True,
 {'shape': {'equal': True, 'shape_a': (52, 36), 'shape_b': (52, 36)},
  'columns': {'different_columns': [],
   'equal': True,
   'only_in_a': [],
   'only_in_b': []},
  'dtypes': {'mismatched_columns': [], 'equal': True},
  'auto_key': 'public_key',
  'cell_compare': {'checked': True,
   'columns_with_differences': [],
   'total_cell_differences': 0,
   'rows_compared': 52,
   'note': "aligned by key='public_key'"},
  'same': True})

In [15]:
df_questionnaire.head(5)

,qnr_seq,variable_name,qtype,question_type,answers,children,condition_expression,hide_if_disabled,featured,instructions,...,cascade_from_question_id,parents,answer_sequence,n_answers,is_linked,parent_1,parent_2,question_sequence,qnr,qnr_version
0,0,,Group,NaN,None,"[{'$type': 'SingleQuestion', 'Answers': [], 'A...",,False,None,None,...,None,,nan,NaN,False,,None,NaN,slbhies_listing,6
1,1,ward,SingleQuestion,0.0,[],[],,False,True,,...,None,Cover,nan,NaN,False,Cover,None,1.0,slbhies_listing,6
2,2,ea,SingleQuestion,0.0,[],[],,False,True,,...,330266f5-d168-b402-a4d3-24921597cd86,Cover,nan,NaN,False,Cover,None,2.0,slbhies_listing,6
3,3,UNITS,Variable,NaN,None,[],None,None,None,None,...,None,Cover,nan,NaN,False,Cover,None,NaN,slbhies_listing,6
4,4,ELIGIBLE,Variable,NaN,None,[],None,None,None,None,...,None,Cover,nan,NaN,False,Cover,None,NaN,slbhies_listing,6


In [16]:
df_questionnaire_kedro.head(5)

,qnr_seq,variable_name,qtype,question_type,answers,children,condition_expression,hide_if_disabled,featured,instructions,...,cascade_from_question_id,parents,answer_sequence,n_answers,is_linked,parent_1,parent_2,question_sequence,qnr,qnr_version
0,0,,Group,NaN,None,"[{'$type': 'SingleQuestion', 'Answers': [], 'A...",,False,None,None,...,None,,nan,NaN,False,,None,NaN,slbhies_listing,6
1,1,ward,SingleQuestion,0.0,[],[],,False,True,,...,None,Cover,nan,NaN,False,Cover,None,1.0,slbhies_listing,6
2,2,ea,SingleQuestion,0.0,[],[],,False,True,,...,330266f5-d168-b402-a4d3-24921597cd86,Cover,nan,NaN,False,Cover,None,2.0,slbhies_listing,6
3,3,UNITS,Variable,NaN,None,[],None,None,None,None,...,None,Cover,nan,NaN,False,Cover,None,NaN,slbhies_listing,6
4,4,ELIGIBLE,Variable,NaN,None,[],None,None,None,None,...,None,Cover,nan,NaN,False,Cover,None,NaN,slbhies_listing,6


In [17]:
df_para_kedro.head(5)

,interview__id,order,event,responsible,role,timestamp_utc,tz_offset,parameters,param,answer,roster_level
0,468fc58b1d4b4196af97bcbfbc5464bb,1,InterviewCreated,WEST_Sup200,1,2024-10-29T01:17:15.712,11:00:00,None,None,None,None
1,468fc58b1d4b4196af97bcbfbc5464bb,2,SupervisorAssigned,WEST_Sup200,1,2024-10-29T01:17:15.712,11:00:00,None,None,None,None
2,468fc58b1d4b4196af97bcbfbc5464bb,3,InterviewModeChanged,WEST_Sup200,1,2024-10-29T01:17:15.712,11:00:00,CAPI||,CAPI,,None
3,468fc58b1d4b4196af97bcbfbc5464bb,4,InterviewerAssigned,WEST_Sup200,1,2024-10-29T01:17:15.712,11:00:00,WEST_Sup200,WEST_Sup200,None,None
4,468fc58b1d4b4196af97bcbfbc5464bb,5,KeyAssigned,None,0,2024-10-29T01:17:15.712,11:00:00,66-54-06-24,66-54-06-24,None,None


In [18]:
df_para.head(5)

,interview__id,order,event,responsible,role,timestamp_utc,tz_offset,parameters,param,answer,...,question_type,answers,question_scope,yes_no_view,is_filtered_combobox,is_integer,cascade_from_question_id,answer_sequence,n_answers,question_sequence
0,468fc58b1d4b4196af97bcbfbc5464bb,1,InterviewCreated,WEST_Sup200,1,2024-10-29 01:17:15.712,0 days 11:00:00,None,None,None,...,NaN,None,NaN,None,None,None,None,nan,NaN,NaN
1,468fc58b1d4b4196af97bcbfbc5464bb,2,SupervisorAssigned,WEST_Sup200,1,2024-10-29 01:17:15.712,0 days 11:00:00,None,None,None,...,NaN,None,NaN,None,None,None,None,nan,NaN,NaN
2,468fc58b1d4b4196af97bcbfbc5464bb,3,InterviewModeChanged,WEST_Sup200,1,2024-10-29 01:17:15.712,0 days 11:00:00,CAPI||,CAPI,,...,NaN,None,NaN,None,None,None,None,nan,NaN,NaN
3,468fc58b1d4b4196af97bcbfbc5464bb,4,InterviewerAssigned,WEST_Sup200,1,2024-10-29 01:17:15.712,0 days 11:00:00,WEST_Sup200,WEST_Sup200,None,...,NaN,None,NaN,None,None,None,None,nan,NaN,NaN
4,468fc58b1d4b4196af97bcbfbc5464bb,5,KeyAssigned,None,0,2024-10-29 01:17:15.712,0 days 11:00:00,66-54-06-24,66-54-06-24,None,...,NaN,None,NaN,None,None,None,None,nan,NaN,NaN


In [ ]:
# meta_old = pq.read_metadata(PROCESSED_DATA_DIR.joinpath("microdata.parquet"))
# meta_new = pq.read_metadata(PROJ_ROOT.joinpath("rissk_kedro", "data", SURVEY, "latest", "30_PROCESSED", "microdata.parquet"))

# print(f"Old Compression: {meta_old.row_group(0).column(0).compression}")
# print(f"New Compression: {meta_new.row_group(0).column(0).compression}")

# print(len(meta_old.metadata[b'pandas']))
# print(len(meta_new.metadata[b'pandas']))

Old Compression: SNAPPY
New Compression: SNAPPY
5387
5405
